# VAST Challenge 2025 MC2 - Data Investigation & Merging

This notebook investigates the two data sources (`Collected_by_the_Government` and `Collected_by_the_Journalist`), documents relationships, compares coverage, and produces merged/cleaned datasets.

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

BASE = Path("../data")
GOV = BASE / "Collected_by_the_Government"
JOUR = BASE / "Collected_by_the_Journalist"
OUT = BASE / "cleaned_data"
OUT.mkdir(exist_ok=True)

CSV_FILES = [
    "meetings", "people", "organizations", "topics", "discussions", "plans", "places", "trips",
    "discussion_people_participations", "discussion_org_participations",
    "plan_people_participations", "plan_org_participations",
    "meeting_discussions", "meeting_plans",
    "discussion_topics", "discussion_plans", "plan_topics",
    "travel_links", "refers_to",
    "trip_people", "trip_places",
]

gov = {name: pd.read_csv(GOV / f"{name}.csv") for name in CSV_FILES}
jour = {name: pd.read_csv(JOUR / f"{name}.csv") for name in CSV_FILES}

print("Loaded all CSV files from both sources.")

Loaded all CSV files from both sources.


## 1. Dataset Dimensions - Side-by-Side Comparison

In [2]:
summary = pd.DataFrame({
    "Table": CSV_FILES,
    "Gov Rows": [len(gov[n]) for n in CSV_FILES],
    "Gov Cols": [gov[n].shape[1] for n in CSV_FILES],
    "Jour Rows": [len(jour[n]) for n in CSV_FILES],
    "Jour Cols": [jour[n].shape[1] for n in CSV_FILES],
})
summary["Row Diff (Jour-Gov)"] = summary["Jour Rows"] - summary["Gov Rows"]
summary["Schema Match"] = summary["Gov Cols"] == summary["Jour Cols"]
summary

,Table,Gov Rows,Gov Cols,Jour Rows,Jour Cols,Row Diff (Jour-Gov),Schema Match
0,meetings,13,3,16,3,3,True
1,people,6,3,6,3,0,True
2,organizations,8,2,8,2,0,True
3,topics,15,3,15,3,0,True
4,discussions,75,3,101,3,26,True
5,plans,55,4,74,4,19,True
6,places,93,6,172,6,79,True
7,trips,194,4,342,4,148,True
8,discussion_people_participations,71,5,99,5,28,True
9,discussion_org_participations,23,5,23,5,0,True


## 2. Entity Table Comparison
Check whether the core entity tables (people, organizations, topics) are identical across both sources.

In [3]:
for name in ["people", "organizations", "topics"]:
    g = gov[name].sort_values(gov[name].columns[0]).reset_index(drop=True)
    j = jour[name].sort_values(jour[name].columns[0]).reset_index(drop=True)
    identical = g.equals(j)
    pk = g.columns[0]
    gov_only = set(g[pk]) - set(j[pk])
    jour_only = set(j[pk]) - set(g[pk])
    print(f"--- {name} ---")
    print(f"  Identical: {identical}")
    print(f"  Gov-only IDs: {gov_only if gov_only else 'none'}")
    print(f"  Jour-only IDs: {jour_only if jour_only else 'none'}")
    print()

--- people ---
  Identical: True
  Gov-only IDs: none
  Jour-only IDs: none

--- organizations ---
  Identical: True
  Gov-only IDs: none
  Jour-only IDs: none

--- topics ---
  Identical: True
  Gov-only IDs: none
  Jour-only IDs: none



## 3. Meeting Coverage Comparison
The Government and Journalist datasets cover different sets of meetings.

In [4]:
gov_meetings = set(gov["meetings"]["meeting_id"])
jour_meetings = set(jour["meetings"]["meeting_id"])

print(f"Government meetings ({len(gov_meetings)}): {sorted(gov_meetings, key=lambda x: int(x.split('_')[1]))}")
print(f"Journalist meetings ({len(jour_meetings)}): {sorted(jour_meetings, key=lambda x: int(x.split('_')[1]))}")
print(f"\nShared: {sorted(gov_meetings & jour_meetings, key=lambda x: int(x.split('_')[1]))}")
print(f"Gov-only: {sorted(gov_meetings - jour_meetings)}")
print(f"Jour-only: {sorted(jour_meetings - gov_meetings, key=lambda x: int(x.split('_')[1]))}")

print("\n--- Meeting details comparison ---")
merged_meetings = pd.merge(
    gov["meetings"].rename(columns={"date": "date_gov", "label": "label_gov"}),
    jour["meetings"].rename(columns={"date": "date_jour", "label": "label_jour"}),
    on="meeting_id", how="outer", indicator=True,
)
merged_meetings["meeting_num"] = merged_meetings["meeting_id"].str.extract(r"(\d+)").astype(int)
merged_meetings.sort_values("meeting_num", inplace=True)
merged_meetings

Government meetings (13): ['Meeting_1', 'Meeting_2', 'Meeting_3', 'Meeting_4', 'Meeting_5', 'Meeting_6', 'Meeting_7', 'Meeting_8', 'Meeting_9', 'Meeting_10', 'Meeting_11', 'Meeting_12', 'Meeting_16']
Journalist meetings (16): ['Meeting_1', 'Meeting_2', 'Meeting_3', 'Meeting_4', 'Meeting_5', 'Meeting_6', 'Meeting_7', 'Meeting_8', 'Meeting_9', 'Meeting_10', 'Meeting_11', 'Meeting_12', 'Meeting_13', 'Meeting_14', 'Meeting_15', 'Meeting_16']

Shared: ['Meeting_1', 'Meeting_2', 'Meeting_3', 'Meeting_4', 'Meeting_5', 'Meeting_6', 'Meeting_7', 'Meeting_8', 'Meeting_9', 'Meeting_10', 'Meeting_11', 'Meeting_12', 'Meeting_16']
Gov-only: []
Jour-only: ['Meeting_13', 'Meeting_14', 'Meeting_15']

--- Meeting details comparison ---


,meeting_id,date_gov,label_gov,date_jour,label_jour,_merge,meeting_num
0,Meeting_1,Meeting 1,Meeting 1,Meeting 1,Meeting 1,both,1
8,Meeting_2,Meeting 2,Meeting 2,Meeting 2,Meeting 2,both,2
9,Meeting_3,Meeting 3,Meeting 3,Meeting 3,Meeting 3,both,3
10,Meeting_4,Meeting 4,Meeting 4,Meeting 4,Meeting 4,both,4
11,Meeting_5,Meeting 5,Meeting 5,Meeting 5,Meeting 5,both,5
12,Meeting_6,Meeting 6,Meeting 6,Meeting 6,Meeting 6,both,6
13,Meeting_7,Meeting 7,Meeting 7,Meeting 7,Meeting 7,both,7
14,Meeting_8,Meeting 8,Meeting 8,Meeting 8,Meeting 8,both,8
15,Meeting_9,Meeting 9,Meeting 9,Meeting 9,Meeting 9,both,9
1,Meeting_10,Meeting 10,Meeting 10,Meeting 10,Meeting 10,both,10


## 4. Discussion & Plan Overlap
Identify which discussions and plans are shared, government-only, or journalist-only.

In [5]:
for name in ["discussions", "plans"]:
    pk = gov[name].columns[0]
    g_ids = set(gov[name][pk])
    j_ids = set(jour[name][pk])
    shared = g_ids & j_ids
    g_only = g_ids - j_ids
    j_only = j_ids - g_ids
    print(f"--- {name} ---")
    print(f"  Gov: {len(g_ids)}, Jour: {len(j_ids)}")
    print(f"  Shared: {len(shared)}, Gov-only: {len(g_only)}, Jour-only: {len(j_only)}")
    if g_only:
        print(f"  Gov-only examples: {list(g_only)[:5]}")
    if j_only:
        print(f"  Jour-only examples: {list(j_only)[:5]}")
    print()

--- discussions ---
  Gov: 75, Jour: 101
  Shared: 75, Gov-only: 0, Jour-only: 26
  Jour-only examples: ['name_harbor_area_Travel_Harbor_Odyssey_Tours_Completed_Discussion', 'name_harbor_area_Travel_The_Bait_Stich_Completed_Discussion', 'renaming_park_himark_Meeting_8_Compile_Nominations_Discussion', 'name_harbor_area_Meeting_11_Harbor_Odyssey_Tours_Discussion', 'waterfront_market_Travel_Harborfront_Market_Completed_Discussion']

--- plans ---
  Gov: 55, Jour: 74
  Shared: 55, Gov-only: 0, Jour-only: 19
  Jour-only examples: ['statue_john_smoth_Meeting_10_Presentation', 'name_inspection_office_Meeting_8_Proposal', 'renaming_park_himark_Meeting_7_Unveiling_Event', 'renaming_park_himark_Travel_Anchor_Way_Grill', 'name_harbor_area_Meeting_14_Finalize_Name']



## 5. Trip Data Overview
The trip data has date inconsistencies (0040 vs 2040 year prefix). Fix and compare.

In [6]:
def normalize_trip_date(df):
    df = df.copy()
    df["date"] = df["date"].str.replace(r"^0040-", "2040-", regex=True)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    return df

gov_trips = normalize_trip_date(gov["trips"])
jour_trips = normalize_trip_date(jour["trips"])

print(f"Government trips: {len(gov_trips)}, date range: {gov_trips['date'].min()} to {gov_trips['date'].max()}")
print(f"Journalist trips: {len(jour_trips)}, date range: {jour_trips['date'].min()} to {jour_trips['date'].max()}")

gov_trip_ids = set(gov_trips["trip_id"])
jour_trip_ids = set(jour_trips["trip_id"])
shared_trips = gov_trip_ids & jour_trip_ids
print(f"\nShared trip IDs: {len(shared_trips)}")
print(f"Gov-only trips: {len(gov_trip_ids - jour_trip_ids)}")
print(f"Jour-only trips: {len(jour_trip_ids - gov_trip_ids)}")

Government trips: 194, date range: 2040-03-31 00:00:00 to 2040-08-02 00:00:00
Journalist trips: 342, date range: 2040-03-31 00:00:00 to 2040-08-02 00:00:00

Shared trip IDs: 194
Gov-only trips: 0
Jour-only trips: 148


In [7]:
for src_name, trips_df in [("Government", gov_trips), ("Journalist", jour_trips)]:
    print(f"\n--- {src_name} trip date distribution ---")
    monthly = trips_df.set_index("date").resample("M").size()
    for dt, cnt in monthly.items():
        print(f"  {dt.strftime('%Y-%m')}: {cnt} trips")


--- Government trip date distribution ---
  2040-03: 1 trips
  2040-04: 46 trips
  2040-05: 44 trips
  2040-06: 52 trips
  2040-07: 48 trips
  2040-08: 3 trips

--- Journalist trip date distribution ---
  2040-03: 1 trips
  2040-04: 79 trips
  2040-05: 85 trips
  2040-06: 90 trips
  2040-07: 82 trips
  2040-08: 5 trips


C:\Users\votev\AppData\Local\Temp\ipykernel_7336\4068529772.py:3: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = trips_df.set_index("date").resample("M").size()
C:\Users\votev\AppData\Local\Temp\ipykernel_7336\4068529772.py:3: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = trips_df.set_index("date").resample("M").size()


## 6. Places Comparison
Compare place coverage between the two sources.

In [8]:
g_place_ids = set(gov["places"]["place_id"].astype(str))
j_place_ids = set(jour["places"]["place_id"].astype(str))

print(f"Gov places: {len(g_place_ids)}, Jour places: {len(j_place_ids)}")
print(f"Shared: {len(g_place_ids & j_place_ids)}")
print(f"Gov-only: {len(g_place_ids - j_place_ids)}")
print(f"Jour-only: {len(j_place_ids - g_place_ids)}")

print("\n--- Zone distribution (Government) ---")
print(gov["places"]["zone"].value_counts())
print("\n--- Zone distribution (Journalist) ---")
print(jour["places"]["zone"].value_counts())

Gov places: 93, Jour places: 172
Shared: 93
Gov-only: 0
Jour-only: 79

--- Zone distribution (Government) ---
zone
commercial     55
government     20
industrial      8
tourism         6
residential     3
connector       1
Name: count, dtype: int64

--- Zone distribution (Journalist) ---
zone
commercial     91
residential    32
government     30
industrial     11
tourism         7
connector       1
Name: count, dtype: int64


## 7. Sentiment Analysis - Who Feels What About Which Topics?
Combine discussion and plan participation data to build a person-level sentiment profile.

In [9]:
def build_sentiment_profile(data, source_label):
    disc_people = data["discussion_people_participations"].copy()
    disc_people = disc_people.merge(
        data["discussion_topics"][["discussion_id", "topic_id"]], on="discussion_id", how="left"
    )
    disc_people["context"] = "discussion"

    plan_people = data["plan_people_participations"].copy()
    plan_people = plan_people.merge(
        data["plan_topics"][["plan_id", "topic_id"]], on="plan_id", how="left"
    )
    plan_people["context"] = "plan"
    plan_people = plan_people.rename(columns={"plan_id": "item_id"})
    disc_people = disc_people.rename(columns={"discussion_id": "item_id"})

    combined = pd.concat([disc_people, plan_people], ignore_index=True)
    combined["source"] = source_label
    return combined

sentiment_gov = build_sentiment_profile(gov, "government")
sentiment_jour = build_sentiment_profile(jour, "journalist")

all_sentiment = pd.concat([sentiment_gov, sentiment_jour], ignore_index=True)
all_sentiment["sentiment"] = pd.to_numeric(all_sentiment["sentiment"], errors="coerce")

person_topic_sentiment = (
    all_sentiment
    .dropna(subset=["sentiment", "topic_id"])
    .groupby(["people_id", "topic_id", "source"])
    .agg(avg_sentiment=("sentiment", "mean"), count=("sentiment", "count"))
    .reset_index()
)

print("Person x Topic x Source sentiment summary:")
person_topic_sentiment.sort_values(["people_id", "topic_id", "source"])

Person x Topic x Source sentiment summary:


,people_id,topic_id,source,avg_sentiment,count
0,Carol Limpet,marine_life_deck,government,0.50,7
1,Carol Limpet,marine_life_deck,journalist,0.50,7
2,Carol Limpet,renaming_park_himark,government,0.50,5
3,Carol Limpet,renaming_park_himark,journalist,0.50,5
4,Carol Limpet,seafood_festival,government,0.75,5
...,...,...,...,...,...
61,Teddy Goldstein,fish_vacuum,journalist,0.50,3
62,Teddy Goldstein,marine_life_deck,government,-0.50,2
63,Teddy Goldstein,marine_life_deck,journalist,-0.50,2
64,Teddy Goldstein,new_crane_lomark,government,1.00,4


In [10]:
print("--- Average sentiment per person (across all topics, both sources) ---")
person_avg = (
    all_sentiment
    .dropna(subset=["sentiment"])
    .groupby("people_id")["sentiment"]
    .agg(["mean", "count", "std"])
    .round(3)
    .sort_values("mean")
)
person_avg

--- Average sentiment per person (across all topics, both sources) ---


,mean,count,std
people_id,,,
Seal,0.108,24,0.050
Teddy Goldstein,0.344,32,0.689
Simone Kat,0.469,84,0.659
Carol Limpet,0.655,42,0.199
Ed Helpsford,0.700,40,0.405
Tante Titan,0.819,54,0.277


## 8. Sentiment Divergence Between Sources
Do the Government and Journalist record the same sentiments for the same (person, topic) pairs?

In [11]:
pivot = person_topic_sentiment.pivot_table(
    index=["people_id", "topic_id"],
    columns="source",
    values="avg_sentiment",
).dropna()

if len(pivot) > 0 and "government" in pivot.columns and "journalist" in pivot.columns:
    pivot["delta"] = (pivot["journalist"] - pivot["government"]).round(3)
    print("Sentiment divergence (journalist - government) for same (person, topic):")
    display(pivot.sort_values("delta"))
    print(f"\nMean absolute divergence: {pivot['delta'].abs().mean():.3f}")
else:
    print("Not enough overlapping data to compare sentiment divergence.")
    print(f"Pivot shape: {pivot.shape}, columns: {list(pivot.columns)}")

Sentiment divergence (journalist - government) for same (person, topic):


source                                   government  journalist  delta
people_id       topic_id                                              
Carol Limpet    marine_life_deck               0.50        0.50    0.0
                renaming_park_himark           0.50        0.50    0.0
                seafood_festival               0.75        0.75    0.0
                waterfront_market              1.00        1.00    0.0
Ed Helpsford    affordable_housing             1.00        1.00    0.0
                concert                        0.50        0.50    0.0
                low_volume_crane               1.00        1.00    0.0
                name_harbor_area               0.00        0.00    0.0
                name_inspection_office         0.00        0.00    0.0
                waterfront_market              1.00        1.00    0.0
Seal            expanding_tourist_wharf        0.10        0.10    0.0
                fish_vacuum                    0.00        0.00    0.0
                low_volume_crane               0.10        0.10    0.0
                statue_john_smoth              0.20        0.20    0.0
Simone Kat      affordable_housing            -1.00       -1.00    0.0
                expanding_tourist_wharf        0.50        0.50    0.0
                fish_vacuum                   -0.10       -0.10    0.0
                heritage_walking_tour          1.00        1.00    0.0
                low_volume_crane               0.75        0.75    0.0
                marine_life_deck               1.00        1.00    0.0
                name_inspection_office         0.00        0.00    0.0
                new_crane_lomark              -0.50       -0.50    0.0
                seafood_festival               0.75        0.75    0.0
                waterfront_market              0.75        0.75    0.0
Teddy Goldstein affordable_housing             1.00        1.00    0.0
                expanding_tourist_wharf       -0.50       -0.50    0.0
                fish_vacuum                    0.50        0.50    0.0
                marine_life_deck              -0.50       -0.50    0.0
                new_crane_lomark               1.00        1.00    0.0


Mean absolute divergence: 0.000


## 9. Organization Sentiment Profiles

In [12]:
def build_org_sentiment(data, source_label):
    disc_org = data["discussion_org_participations"].copy()
    disc_org = disc_org.merge(
        data["discussion_topics"][["discussion_id", "topic_id"]], on="discussion_id", how="left"
    )
    plan_org = data["plan_org_participations"].copy()
    plan_org = plan_org.merge(
        data["plan_topics"][["plan_id", "topic_id"]], on="plan_id", how="left"
    )
    disc_org["context"] = "discussion"
    plan_org["context"] = "plan"
    combined = pd.concat([disc_org, plan_org], ignore_index=True)
    combined["source"] = source_label
    combined["sentiment"] = pd.to_numeric(combined["sentiment"], errors="coerce")
    return combined

org_sent_gov = build_org_sentiment(gov, "government")
org_sent_jour = build_org_sentiment(jour, "journalist")
all_org_sent = pd.concat([org_sent_gov, org_sent_jour], ignore_index=True)

org_avg = (
    all_org_sent.dropna(subset=["sentiment"])
    .groupby(["organization_id", "topic_id"])["sentiment"]
    .agg(["mean", "count"])
    .round(3)
    .sort_values(["organization_id", "mean"])
)
print("Organization sentiment by topic:")
org_avg

Organization sentiment by topic:


mean  count
organization_id          topic_id                            
Builders Association     affordable_housing        1.0      8
Daughters of Port Grove  name_harbor_area          0.0      4
                         heritage_walking_tour     0.5      4
                         statue_john_smoth         1.0      4
High Seas Fishing Inc.   name_inspection_office    0.0      4
                         fish_vacuum               1.0      8
Industrial Shipping      affordable_housing        1.0      8
                         new_crane_lomark          1.0      4
PTA                      renaming_park_himark      0.0      4
Paackland Container Inc. expanding_tourist_wharf  -0.5      4
Saltwater Serenades      affordable_housing       -1.0      4
                         new_crane_lomark         -1.0      4
Tours Central Ticketing  new_crane_lomark         -1.0      4
                         expanding_tourist_wharf   1.0      4

## 10. Topic Activity Timeline
Which topics are discussed in which meetings? Build a meeting x topic matrix.

In [13]:
def topic_timeline(data, source):
    md = data["meeting_discussions"].merge(
        data["discussion_topics"][["discussion_id", "topic_id"]], on="discussion_id"
    )
    md["source"] = source
    return md

tl_gov = topic_timeline(gov, "government")
tl_jour = topic_timeline(jour, "journalist")
tl_all = pd.concat([tl_gov, tl_jour], ignore_index=True)

tl_all["meeting_num"] = tl_all["meeting_id"].str.extract(r"(\d+)").astype(int)

topic_meeting_matrix = (
    tl_all.drop_duplicates(subset=["meeting_id", "topic_id"])
    .pivot_table(index="topic_id", columns="meeting_num", aggfunc="size", fill_value=0)
)
topic_meeting_matrix = topic_meeting_matrix.reindex(columns=sorted(topic_meeting_matrix.columns))

print("Topic x Meeting matrix (1 = discussed in that meeting, across both sources):")
topic_meeting_matrix.clip(upper=1)

Topic x Meeting matrix (1 = discussed in that meeting, across both sources):


meeting_num,1,2,3,4,5,6,7,8,9,10,11,12,14,16
topic_id,,,,,,,,,,,,,,
affordable_housing,0,0,0,0,0,1,1,1,1,1,0,0,0,0
concert,0,0,0,0,0,0,1,1,0,0,0,0,0,1
deep_fishing_dock,0,1,1,1,0,0,0,0,0,0,0,0,0,0
expanding_tourist_wharf,0,0,0,0,0,0,1,1,1,1,1,0,0,0
fish_vacuum,1,1,1,0,0,0,0,0,0,0,0,0,0,0
heritage_walking_tour,0,0,0,0,0,0,1,1,1,0,0,0,0,0
low_volume_crane,0,0,1,0,1,0,0,1,1,0,0,0,0,0
marine_life_deck,0,0,0,0,0,0,0,0,0,1,1,1,0,0
name_harbor_area,0,0,0,0,0,0,0,0,0,1,1,1,1,0


## 11. Trip-People-Places Network
Who travels where, and how often?

In [14]:
def build_trip_summary(data, source):
    trips = data["trips"].copy()
    trips["date"] = trips["date"].str.replace(r"^0040-", "2040-", regex=True)
    tp = data["trip_people"].copy()
    tpl = data["trip_places"].copy()
    merged = tp.merge(tpl, on="trip_id", suffixes=("_person", "_place"))
    merged = merged.merge(trips[["trip_id", "date", "start_time", "end_time"]], on="trip_id")
    merged["source"] = source
    return merged

trip_sum_gov = build_trip_summary(gov, "government")
trip_sum_jour = build_trip_summary(jour, "journalist")

print(f"Gov trip-person-place records: {len(trip_sum_gov)}")
print(f"Jour trip-person-place records: {len(trip_sum_jour)}")

print("\n--- Person trip frequency (Government) ---")
print(gov["trip_people"]["people_id"].value_counts())
print("\n--- Person trip frequency (Journalist) ---")
print(jour["trip_people"]["people_id"].value_counts())

Gov trip-person-place records: 234
Jour trip-person-place records: 1363

--- Person trip frequency (Government) ---
people_id
Carol Limpet       69
Simone Kat         67
Seal               53
Tante Titan         4
Teddy Goldstein     1
Name: count, dtype: int64

--- Person trip frequency (Journalist) ---
people_id
Carol Limpet       69
Simone Kat         67
Ed Helpsford       61
Seal               53
Tante Titan        49
Teddy Goldstein    43
Name: count, dtype: int64


## 12. Plan Type Distribution

In [15]:
print("--- Government plan types ---")
print(gov["plans"]["plan_type"].str.lower().value_counts())
print("\n--- Journalist plan types ---")
print(jour["plans"]["plan_type"].str.lower().value_counts())

--- Government plan types ---
plan_type
travel          16
report          10
proposal        10
feedback         6
discussion       5
presentation     4
take action      4
Name: count, dtype: int64

--- Journalist plan types ---
plan_type
travel          22
proposal        14
report          12
feedback         9
presentation     6
take action      6
discussion       5
Name: count, dtype: int64


## 13. Discussion Status Tracking
What is the status progression of discussions about plans?

In [16]:
for label, data in [("Government", gov), ("Journalist", jour)]:
    dp = data["discussion_plans"]
    print(f"--- {label} discussion_plans status ---")
    print(dp["status"].value_counts(dropna=False))
    print()

--- Government discussion_plans status ---
status
completed      30
planned        30
in_progress     6
introduced      3
Completed       1
Name: count, dtype: int64

--- Journalist discussion_plans status ---
status
completed      45
planned        38
in_progress     8
introduced      4
Completed       1
Name: count, dtype: int64



---
# Data Merging & Cleaning

Below we merge both sources into unified tables and output them to `data/cleaned_data/`.

## 14. Merge Strategy

- **Entity tables** (people, organizations, topics): identical in both sources -> use as-is
- **Meetings**: union (journalist has 3 extra meetings)
- **Discussions, plans**: union, deduplicate by primary key
- **Junction tables** (participations, links): union with a `source` column, deduplicate where records match exactly
- **Trips, trip_people, trip_places**: union with `source` column since trip IDs overlap with different data
- **Places**: union, deduplicate by place_id
- **Date normalization**: fix 0040 -> 2040 in trip dates

In [17]:
# --- Entity tables (identical) ---
for name in ["people", "organizations", "topics"]:
    gov[name].to_csv(OUT / f"{name}.csv", index=False)
    print(f"Saved {name}.csv ({len(gov[name])} rows)")

Saved people.csv (6 rows)
Saved organizations.csv (8 rows)
Saved topics.csv (15 rows)


In [18]:
# --- Meetings: union ---
meetings_merged = pd.concat([gov["meetings"], jour["meetings"]], ignore_index=True)
meetings_merged = meetings_merged.drop_duplicates(subset=["meeting_id"]).sort_values(
    "meeting_id", key=lambda s: s.str.extract(r"(\d+)")[0].astype(int)
).reset_index(drop=True)
meetings_merged.to_csv(OUT / "meetings.csv", index=False)
print(f"Saved meetings.csv ({len(meetings_merged)} rows)")
meetings_merged

Saved meetings.csv (16 rows)


,meeting_id,date,label
0,Meeting_1,Meeting 1,Meeting 1
1,Meeting_2,Meeting 2,Meeting 2
2,Meeting_3,Meeting 3,Meeting 3
3,Meeting_4,Meeting 4,Meeting 4
4,Meeting_5,Meeting 5,Meeting 5
5,Meeting_6,Meeting 6,Meeting 6
6,Meeting_7,Meeting 7,Meeting 7
7,Meeting_8,Meeting 8,Meeting 8
8,Meeting_9,Meeting 9,Meeting 9
9,Meeting_10,Meeting 10,Meeting 10


In [19]:
# --- Discussions & Plans: union, dedup by PK ---
for name, pk in [("discussions", "discussion_id"), ("plans", "plan_id")]:
    merged = pd.concat([gov[name], jour[name]], ignore_index=True)
    merged = merged.drop_duplicates(subset=[pk]).reset_index(drop=True)
    merged.to_csv(OUT / f"{name}.csv", index=False)
    print(f"Saved {name}.csv ({len(merged)} rows, from gov={len(gov[name])} + jour={len(jour[name])})")

Saved discussions.csv (101 rows, from gov=75 + jour=101)
Saved plans.csv (74 rows, from gov=55 + jour=74)


In [20]:
# --- Places: union, dedup by place_id ---
places_merged = pd.concat([gov["places"], jour["places"]], ignore_index=True)
places_merged["place_id"] = places_merged["place_id"].astype(str)
places_merged = places_merged.drop_duplicates(subset=["place_id"]).reset_index(drop=True)
places_merged.to_csv(OUT / "places.csv", index=False)
print(f"Saved places.csv ({len(places_merged)} rows)")

Saved places.csv (172 rows)

In [21]:
# --- Junction tables: union with source tag, then dedup on natural key ---
junction_configs = {
    "meeting_discussions": ["meeting_id", "discussion_id"],
    "meeting_plans": ["meeting_id", "plan_id"],
    "discussion_topics": ["discussion_id", "topic_id"],
    "discussion_plans": ["discussion_id", "plan_id"],
    "plan_topics": ["plan_id", "topic_id"],
    "travel_links": ["plan_id", "place_id"],
    "refers_to": ["discussion_id", "place_id"],
}

for name, key_cols in junction_configs.items():
    g = gov[name].copy()
    j = jour[name].copy()
    g["source"] = "government"
    j["source"] = "journalist"
    merged = pd.concat([g, j], ignore_index=True)
    merged = merged.drop_duplicates(subset=key_cols).reset_index(drop=True)
    merged.to_csv(OUT / f"{name}.csv", index=False)
    print(f"Saved {name}.csv ({len(merged)} rows)")

Saved meeting_discussions.csv (101 rows)
Saved meeting_plans.csv (74 rows)
Saved discussion_topics.csv (102 rows)
Saved discussion_plans.csv (96 rows)
Saved plan_topics.csv (73 rows)


Saved travel_links.csv (22 rows)
Saved refers_to.csv (43 rows)


In [22]:
# --- Participation tables: keep source tag, dedup on full record ---
participation_tables = [
    "discussion_people_participations",
    "discussion_org_participations",
    "plan_people_participations",
    "plan_org_participations",
]

for name in participation_tables:
    g = gov[name].copy()
    j = jour[name].copy()
    g["source"] = "government"
    j["source"] = "journalist"
    merged = pd.concat([g, j], ignore_index=True)
    key_cols = [c for c in merged.columns if c not in ["source"]]
    merged = merged.drop_duplicates(subset=key_cols).reset_index(drop=True)
    merged.to_csv(OUT / f"{name}.csv", index=False)
    print(f"Saved {name}.csv ({len(merged)} rows)")

Saved discussion_people_participations.csv (99 rows)
Saved discussion_org_participations.csv (23 rows)
Saved plan_people_participations.csv (75 rows)
Saved plan_org_participations.csv (17 rows)


In [23]:
# --- Trips: union with source, normalize dates ---
for name in ["trips", "trip_people", "trip_places"]:
    g = gov[name].copy()
    j = jour[name].copy()
    g["source"] = "government"
    j["source"] = "journalist"
    merged = pd.concat([g, j], ignore_index=True)
    if "date" in merged.columns:
        merged["date"] = merged["date"].str.replace(r"^0040-", "2040-", regex=True)
    if "time" in merged.columns:
        merged["time"] = merged["time"].astype(str).str.replace(r"^0040-", "2040-", regex=True)
    merged = merged.drop_duplicates().reset_index(drop=True)
    merged.to_csv(OUT / f"{name}.csv", index=False)
    print(f"Saved {name}.csv ({len(merged)} rows)")

Saved trips.csv (536 rows)
Saved trip_people.csv (536 rows)
Saved trip_places.csv (1597 rows)


## 15. Consolidated Wide Tables
Create denormalized "wide" tables for easy analysis.

In [24]:
# --- Wide discussion table: discussion + meeting + topics + people sentiments ---

disc_all = pd.read_csv(OUT / "discussions.csv")
disc_topics = pd.read_csv(OUT / "discussion_topics.csv")
mtg_disc = pd.read_csv(OUT / "meeting_discussions.csv")
disc_people_part = pd.read_csv(OUT / "discussion_people_participations.csv")
topics_all = pd.read_csv(OUT / "topics.csv")

wide_disc = disc_all.copy()

# Add topic
topic_map = disc_topics.groupby("discussion_id")["topic_id"].apply(lambda x: "; ".join(x)).reset_index()
topic_map.columns = ["discussion_id", "topics"]
wide_disc = wide_disc.merge(topic_map, on="discussion_id", how="left")

# Add meeting
mtg_map = mtg_disc.groupby("discussion_id")["meeting_id"].apply(lambda x: "; ".join(x)).reset_index()
mtg_map.columns = ["discussion_id", "meetings"]
wide_disc = wide_disc.merge(mtg_map, on="discussion_id", how="left")

# Add average sentiment across all people
disc_people_part["sentiment"] = pd.to_numeric(disc_people_part["sentiment"], errors="coerce")
sent_agg = disc_people_part.groupby("discussion_id").agg(
    avg_sentiment=("sentiment", "mean"),
    num_participants=("people_id", "nunique"),
    participants=("people_id", lambda x: "; ".join(sorted(x.unique()))),
).reset_index().round(3)
wide_disc = wide_disc.merge(sent_agg, on="discussion_id", how="left")

wide_disc.to_csv(OUT / "wide_discussions.csv", index=False)
print(f"Saved wide_discussions.csv ({len(wide_disc)} rows, {wide_disc.shape[1]} columns)")
wide_disc.head(10)

Saved wide_discussions.csv (101 rows, 8 columns)


,discussion_id,short_title,long_title,topics,meetings,avg_sentiment,num_participants,participants
0,concert_Meeting_7_Discussion,concert_Meeting_7_Discussion,Discuss any issues with concert. Discuss needs...,concert,Meeting_7,0.5,2.0,Ed Helpsford; Tante Titan
1,affordable_housing_Meeting_9_Invite_Developers...,affordable_housing_Meeting_9_Invite_Developers...,Invite housing developers,affordable_housing,Meeting_9,NaN,NaN,NaN
2,low_volume_crane_Meeting_3_Proposal_Discussion,low_volume_crane_Meeting_3_Proposal_Discussion,Discuss potential benefits of low-volume unloa...,low_volume_crane,Meeting_3,1.0,1.0,Ed Helpsford
3,affordable_housing_Meeting_10_Travel_Findings_...,affordable_housing_Meeting_10_Travel_Findings_...,Report on travel findings and draft housing pr...,affordable_housing,Meeting_10,1.0,1.0,Ed Helpsford
4,waterfront_market_Meeting_9_Report_Discussion,waterfront_market_Meeting_9_Report_Discussion,"Plan a detailed plan for the market, including...",waterfront_market,Meeting_9,1.0,1.0,Ed Helpsford
5,low_volume_crane_Travel_Harbor_Route_Solutions...,low_volume_crane_Travel_Harbor_Route_Solutions...,Discuss the completed travel plans to Harbor R...,low_volume_crane,Meeting_8,0.1,1.0,Seal
6,affordable_housing_Meeting_6_Sites_Proposal_Di...,affordable_housing_Meeting_6_Sites_Proposal_Di...,Discuss potential sites for affordable housing,affordable_housing,Meeting_6,NaN,NaN,NaN
7,fish_vacuum_Meeting_3_Report_Discussion,fish_vacuum_Meeting_3_Report_Discussion,Provide report on cost-benefit analysis,fish_vacuum,Meeting_3,0.5,1.0,Teddy Goldstein
8,affordable_housing_Travel_Waveside_Townhomes_C...,affordable_housing_Travel_Waveside_Townhomes_C...,Discuss the completed travel plans to Waveside...,affordable_housing,Meeting_8,1.0,1.0,Ed Helpsford
9,fish_vacuum_Meeting_2_Cost_Benefit_Analysis_Di...,fish_vacuum_Meeting_2_Cost_Benefit_Analysis_Di...,Provide cost-benefit analysis of implementing ...,fish_vacuum,Meeting_2,NaN,NaN,NaN


In [25]:
# --- Wide plans table ---

plans_all = pd.read_csv(OUT / "plans.csv")
plan_topics_df = pd.read_csv(OUT / "plan_topics.csv")
mtg_plans = pd.read_csv(OUT / "meeting_plans.csv")
plan_people_part = pd.read_csv(OUT / "plan_people_participations.csv")

wide_plans = plans_all.copy()

# Add topic
pt_map = plan_topics_df.groupby("plan_id")["topic_id"].apply(lambda x: "; ".join(x)).reset_index()
pt_map.columns = ["plan_id", "topics"]
wide_plans = wide_plans.merge(pt_map, on="plan_id", how="left")

# Add meeting
mp_map = mtg_plans.groupby("plan_id")["meeting_id"].apply(lambda x: "; ".join(x)).reset_index()
mp_map.columns = ["plan_id", "meetings"]
wide_plans = wide_plans.merge(mp_map, on="plan_id", how="left")

# Add sentiment
plan_people_part["sentiment"] = pd.to_numeric(plan_people_part["sentiment"], errors="coerce")
plan_sent = plan_people_part.groupby("plan_id").agg(
    avg_sentiment=("sentiment", "mean"),
    num_participants=("people_id", "nunique"),
    participants=("people_id", lambda x: "; ".join(sorted(x.unique()))),
).reset_index().round(3)
wide_plans = wide_plans.merge(plan_sent, on="plan_id", how="left")

wide_plans.to_csv(OUT / "wide_plans.csv", index=False)
print(f"Saved wide_plans.csv ({len(wide_plans)} rows, {wide_plans.shape[1]} columns)")
wide_plans.head(10)

Saved wide_plans.csv (74 rows, 9 columns)


,plan_id,short_title,long_title,plan_type,topics,meetings,avg_sentiment,num_participants,participants
0,marine_life_deck_Meeting_12_Environmental_Impa...,marine_life_deck_Meeting_12_Environmental_Impa...,Present findings from the environmental impact...,Report,marine_life_deck,Meeting_12,-0.500,1.0,Teddy Goldstein
1,low_volume_crane_Meeting_8_Report,low_volume_crane_Meeting_8_Report,Report on travel and submit letter of support ...,report,low_volume_crane,Meeting_8,0.100,1.0,Seal
2,deep_fishing_dock_Meeting_3_Maintenance_Plan,deep_fishing_dock_Meeting_3_Maintenance_Plan,Discuss maintenance plan and designate travel ...,proposal,deep_fishing_dock,Meeting_3,NaN,1.0,Teddy Goldstein
3,affordable_housing_Travel_Waveside_Townhomes,affordable_housing_Travel_Waveside_Townhomes,Travel to Waveside Townhomes in Lomark,Travel,affordable_housing,Meeting_7,-1.000,1.0,Simone Kat
4,affordable_housing_Travel_Tidewater_Flats,affordable_housing_Travel_Tidewater_Flats,Travel to Tidewater Flats in Lomark,Travel,affordable_housing,Meeting_9,1.000,1.0,Teddy Goldstein
5,waterfront_market_Meeting_7_Benefits_Feasibility,waterfront_market_Meeting_7_Benefits_Feasibility,Discuss the benefits and feasibility of establ...,Discussion,waterfront_market,Meeting_7,0.625,2.0,Ed Helpsford; Tante Titan
6,name_inspection_office_Meeting_8_Importance,name_inspection_office_Meeting_8_Importance,Discuss the importance of naming the inspectio...,presentation,name_inspection_office,Meeting_8,NaN,NaN,NaN
7,concert_Meeting_16_Road_Closures,concert_Meeting_16_Road_Closures,Status of road closures and traffic issues for...,Report,concert,Meeting_16,0.500,1.0,Ed Helpsford
8,new_crane_lomark_Meeting_5_Cost_Impact_Report,new_crane_lomark_Meeting_5_Cost_Impact_Report,Get a formal cost impact & environment report,report,new_crane_lomark,Meeting_5,1.000,1.0,Teddy Goldstein
9,new_crane_lomark_Meeting_8_Share_Findings,new_crane_lomark_Meeting_8_Share_Findings,Share findings and plan next steps,report,new_crane_lomark,Meeting_8,1.000,1.0,Teddy Goldstein


In [26]:
# --- Wide trips table: trip + people + places ---

trips_all = pd.read_csv(OUT / "trips.csv")
tp_all = pd.read_csv(OUT / "trip_people.csv")
tpl_all = pd.read_csv(OUT / "trip_places.csv")
places_all = pd.read_csv(OUT / "places.csv")

people_per_trip = tp_all.groupby("trip_id")["people_id"].apply(lambda x: "; ".join(sorted(x.unique()))).reset_index()
people_per_trip.columns = ["trip_id", "travelers"]

tpl_all["place_id"] = tpl_all["place_id"].astype(str)
places_all["place_id"] = places_all["place_id"].astype(str)
tpl_named = tpl_all.merge(places_all[["place_id", "name", "zone"]], on="place_id", how="left")
places_per_trip = tpl_named.groupby("trip_id").agg(
    places_visited=("name", lambda x: "; ".join([str(v) for v in x.unique() if pd.notna(v)])),
    zones=("zone", lambda x: "; ".join([str(v) for v in x.unique() if pd.notna(v)])),
    num_stops=("place_id", "nunique"),
).reset_index()

wide_trips = trips_all.merge(people_per_trip, on="trip_id", how="left")
wide_trips = wide_trips.merge(places_per_trip, on="trip_id", how="left")

wide_trips.to_csv(OUT / "wide_trips.csv", index=False)
print(f"Saved wide_trips.csv ({len(wide_trips)} rows, {wide_trips.shape[1]} columns)")
wide_trips.head(10)

Saved wide_trips.csv (536 rows, 9 columns)


,trip_id,date,start_time,end_time,source,travelers,places_visited,zones,num_stops
0,trip_146,2040-04-06,06:09:00,07:36:00,government,Simone Kat,Port Grove Customs House; Suna Spit; Himark Ci...,government; connector,5
1,trip_188,2040-06-20,09:00:00,21:00:00,government,Carol Limpet,Haacklee Ferry Terminal; South Paackland Ferry...,government; commercial,7
2,trip_33,2040-06-04,07:00:00,19:00:00,government,Tante Titan,South Paackland Ferry Terminal; Haacklee Ferry...,government,3
3,trip_47,2040-04-10,06:38:00,09:31:00,government,Simone Kat,Port Grove Customs House; Suna Spit; Himark Cu...,government; connector; tourism,5
4,trip_177,2040-04-01,09:00:00,21:00:00,government,Carol Limpet,Lomark Ferry Terminal; Paackland Ferry Termina...,government; tourism; commercial,6
5,trip_274,2040-05-10,07:06:00,08:57:00,government,Simone Kat,Port Authority Office,government; industrial,2
6,trip_20,2040-05-27,06:07:00,13:00:00,government,Simone Kat,Port Grove Customs House; Haacklee Ferry Terminal,government,2
7,trip_65,2040-05-23,06:19:00,08:27:00,government,Simone Kat,Port Grove Customs House; The Bait & Stich; Ha...,government; tourism,3
8,trip_114,2040-05-14,07:31:00,10:21:00,government,Simone Kat,Lomark Post Office,government,1
9,trip_227,2040-07-16,07:28:00,08:00:00,government,Tante Titan,Paackland City Hall,government,1


In [27]:
# --- Person sentiment matrix: person x topic with avg sentiment ---

all_participation = pd.concat([
    sentiment_gov[["people_id", "topic_id", "sentiment", "source"]],
    sentiment_jour[["people_id", "topic_id", "sentiment", "source"]],
], ignore_index=True)

all_participation["sentiment"] = pd.to_numeric(all_participation["sentiment"], errors="coerce")

person_topic_matrix = all_participation.dropna(subset=["sentiment", "topic_id"]).pivot_table(
    index="people_id", columns="topic_id", values="sentiment", aggfunc="mean"
).round(3)

person_topic_matrix.to_csv(OUT / "person_topic_sentiment_matrix.csv")
print(f"Saved person_topic_sentiment_matrix.csv")
person_topic_matrix

Saved person_topic_sentiment_matrix.csv


topic_id,affordable_housing,concert,expanding_tourist_wharf,fish_vacuum,heritage_walking_tour,low_volume_crane,marine_life_deck,name_harbor_area,name_inspection_office,new_crane_lomark,renaming_park_himark,seafood_festival,statue_john_smoth,waterfront_market
people_id,,,,,,,,,,,,,,
Carol Limpet,NaN,NaN,NaN,NaN,NaN,NaN,0.5,NaN,NaN,NaN,0.5,0.75,NaN,1.00
Ed Helpsford,1.0,0.5,NaN,NaN,NaN,1.00,NaN,0.0,0.0,NaN,NaN,NaN,NaN,1.00
Seal,NaN,NaN,0.1,0.0,NaN,0.10,NaN,NaN,NaN,NaN,NaN,NaN,0.2,NaN
Simone Kat,-1.0,NaN,0.5,-0.1,1.0,0.75,1.0,NaN,0.0,-0.5,NaN,0.75,NaN,0.75
Tante Titan,NaN,0.5,NaN,NaN,1.0,NaN,NaN,1.0,1.0,NaN,1.0,0.75,1.0,0.25
Teddy Goldstein,1.0,NaN,-0.5,0.5,NaN,NaN,-0.5,NaN,NaN,1.0,NaN,NaN,NaN,NaN


## 16. Industry Involvement Summary
Which industries are involved in each topic?

In [28]:
industry_data = all_sentiment.dropna(subset=["industry", "topic_id"]).copy()
industry_data["industry"] = industry_data["industry"].astype(str)

industry_topic = industry_data.groupby("topic_id")["industry"].apply(
    lambda x: "; ".join(sorted(set(x)))
).reset_index()
industry_topic.columns = ["topic_id", "industries_involved"]

industry_topic = industry_topic.merge(topics_all, on="topic_id", how="left")
industry_topic.to_csv(OUT / "topic_industry_summary.csv", index=False)
print("Saved topic_industry_summary.csv")
industry_topic

Saved topic_industry_summary.csv


,topic_id,industries_involved,short_topic,long_topic
0,affordable_housing,"['large vessel', 'small vessel']",affordable_housing,Affordable housing for fishing workers (confli...
1,concert,['tourism'],concert,Plan for Concert
2,expanding_tourist_wharf,['tourism'],expanding_tourist_wharf,Expanding the tourist wharf/infrastructure in ...
3,fish_vacuum,['large vessel'],fish_vacuum,Fish Vacuum
4,heritage_walking_tour,['tourism'],heritage_walking_tour,Developing a Heritage Walking Tour in Haacklee
5,low_volume_crane,['small vessel'],low_volume_crane,Low-volume unload crane in Haacklee
6,marine_life_deck,['tourism'],marine_life_deck,Establishing a Marine Life Observation Deck in...
7,name_harbor_area,[],name_harbor_area,Putting a name on pictureque harbor area in Po...
8,name_inspection_office,[],name_inspection_office,Putting a name on the inspection office in Lomark
9,new_crane_lomark,['large vessel'],new_crane_lomark,Money for a new crane at Lomark


## 17. Final Output Inventory

In [29]:
print("=== Files in cleaned_data/ ===")
for f in sorted(OUT.glob("*.csv")):
    df = pd.read_csv(f)
    print(f"  {f.name:50s} {df.shape[0]:>5d} rows x {df.shape[1]:>2d} cols")

=== Files in cleaned_data/ ===


  discussion_org_participations.csv                     23 rows x  6 cols
  discussion_people_participations.csv                  99 rows x  6 cols
  discussion_plans.csv                                  96 rows x  4 cols
  discussion_topics.csv                                102 rows x  4 cols
  discussions.csv                                      101 rows x  3 cols
  meeting_discussions.csv                              101 rows x  3 cols
  meeting_plans.csv                                     74 rows x  3 cols
  meetings.csv                                          16 rows x  3 cols
  organizations.csv                                      8 rows x  2 cols
  people.csv                                             6 rows x  3 cols
  person_topic_sentiment_matrix.csv                      6 rows x 15 cols
  places.csv                                           172 rows x  6 cols
  plan_org_participations.csv                           17 rows x  6 cols
  plan_people_participations.csv      

  plan_topics.csv                                       73 rows x  3 cols
  plans.csv                                             74 rows x  4 cols
  refers_to.csv                                         43 rows x  3 cols
  topic_industry_summary.csv                            14 rows x  4 cols
  topics.csv                                            15 rows x  3 cols
  travel_links.csv                                      22 rows x  3 cols
  trip_people.csv                                      536 rows x  4 cols
  trip_places.csv                                     1597 rows x  4 cols
  trips.csv                                            536 rows x  5 cols
  wide_discussions.csv                                 101 rows x  8 cols
  wide_plans.csv                                        74 rows x  9 cols
  wide_trips.csv                                       536 rows x  9 cols


## Summary of Findings

### Dataset Structure
Both sources share an **identical relational schema** with 8 entity tables and 13 junction tables, built around a committee that manages community topics through meetings, discussions, plans, and travel.

### Key Differences Between Sources
| Dimension | Government | Journalist |
|-----------|-----------|------------|
| Meetings | 13 (1-12, 16) | 16 (1-16) |
| Discussions | 75 | 101 |
| Plans | 55 | 74 |
| Trips | 194 | 342 |
| Places | 93 | 172 |
| People participation records | 71 disc + 49 plan | 99 disc + 75 plan |

### What's Identical
- **People** (6 committee members), **Organizations** (8), **Topics** (15) are the same in both
- Organization and people participation sentiments are largely consistent

### What's Different
- **Journalist has broader coverage**: 3 extra meetings (13-15), more discussions, plans, trips, and places
- **Trip data has date bugs**: Some use year `0040`, others `2040` (normalized in cleaned output)
- **Different people tracked per discussion**: Each source picks different participants for certain discussions

### Cleaned Output
- **Normalized tables**: All junction tables merged with `source` column
- **Wide tables**: `wide_discussions.csv`, `wide_plans.csv`, `wide_trips.csv` with denormalized joins
- **Sentiment matrix**: `person_topic_sentiment_matrix.csv` - 6 people x 15 topics
- **Industry summary**: `topic_industry_summary.csv` - which industries care about which topics